# Dataset Exploration & Preprocessing

Explores all 8 multilingual datasets downloaded to `data/`. For each dataset we look at:
- Schema (columns, dtypes, row counts)
- Sample rows
- Label / answer distributions
- Preprocessing into a **unified format** for downstream experiments

**Unified schema** (one row = one example):
```
dataset    : str   — dataset name
language   : str   — language code / name
task_type  : str   — math_reasoning | commonsense | nli | qa | mcqa
id         : str   — unique example id
input      : str   — full prompt-ready input text
choices    : list  — answer options (None for open-ended tasks)
label      : str   — ground-truth answer / label
```

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path("data")

# ── helpers ──────────────────────────────────────────────────────────────────
def load(dataset: str, language: str) -> pd.DataFrame:
    path = DATA_DIR / dataset / language / "dataset.parquet"
    return pd.read_parquet(path)

def show(df: pd.DataFrame, n: int = 2) -> None:
    """Pretty-print a few rows without truncating long strings."""
    with pd.option_context("display.max_colwidth", 120):
        display(df.head(n))

print("Ready.")

Ready.


## Manifest overview

In [2]:
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(f"Generated at : {manifest['generated_at']}")
print(f"Total files  : {manifest['total_files']}")

manifest_df = pd.DataFrame(manifest["files"])
summary = (
    manifest_df.groupby("dataset")
    .agg(languages=("language", "count"), total_mb=("size_mb", "sum"), total_examples=("examples", "sum"))
    .rename(columns={"languages": "# langs", "total_mb": "MB", "total_examples": "examples"})
    .round(2)
)
display(summary)

Generated at : 2026-02-25T18:53:27.873696+00:00
Total files  : 85


,# langs,MB,examples
dataset,,,
global_mmlu_lite,18,2.56,3200.0
mgsm,11,0.51,0.0
ml_nli_26lang,1,0.27,0.0
mlqa,7,24.76,0.0
tydiqa,11,30.40,0.0
xcopa,11,0.47,0.0
xnli,15,0.46,0.0
xstorycloze,11,1.55,0.0


---
## 1. MGSM — Multilingual Grade School Math
**Task**: Math word-problem reasoning  
**Languages**: en es fr de ru zh ja th sw bn te (11)  
**Split**: test (250 rows × language)

In [3]:
mgsm_en = load("mgsm", "en")
print("Shape:", mgsm_en.shape)
print("Columns:", list(mgsm_en.columns))
print("\nNull counts:")
print(mgsm_en.isnull().sum())
show(mgsm_en)

Shape: (250, 4)
Columns: ['question', 'answer', 'answer_number', 'equation_solution']

Null counts:
question               0
answer               250
answer_number          0
equation_solution    250
dtype: int64


,question,answer,answer_number,equation_solution
0,Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends ever...,NaN,18,NaN
1,A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?,NaN,3,NaN


In [4]:
print("answer_number stats:")
print(mgsm_en["answer_number"].describe())
print("\nSample question→answer:")
for _, row in mgsm_en.head(3).iterrows():
    print(f"  Q: {row['question'][:90]}...")
    print(f"  A: {row['answer_number']}")
    print()

answer_number stats:
count       250.000000
mean       3179.952000
std       20682.587312
min           1.000000
25%          16.000000
50%          48.000000
75%         162.250000
max      276000.000000
Name: answer_number, dtype: float64

Sample question→answer:
  Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes mu...
  A: 18

  Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in tota...
  A: 3

  Q: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,00...
  A: 70000



---
## 2. XCOPA — Cross-lingual Choice Of Plausible Alternatives
**Task**: Commonsense causal reasoning (cause/effect)  
**Languages**: et ht id it qu sw ta th tr vi zh (11)  
**Split**: test (500 rows × language)  
**Label**: 0 = choice1 is correct, 1 = choice2 is correct

In [5]:
xcopa_id = load("xcopa", "id")   # Indonesian — readable example
print("Shape:", xcopa_id.shape)
print("Columns:", list(xcopa_id.columns))
show(xcopa_id)
print("Label distribution:")
print(xcopa_id["label"].value_counts().to_string())
print("\nQuestion types:")
print(xcopa_id["question"].value_counts().to_string())

Shape: (500, 7)
Columns: ['premise', 'choice1', 'choice2', 'question', 'label', 'idx', 'changed']


,premise,choice1,choice2,question,label,idx,changed
0,Barang itu dikemas dalam bungkus gelembung.,Barang itu rapuh.,Barang itu kecil.,cause,0,0,False
1,Saya kosongkan kantong saya.,Saya ambil sebuah potongan tiket.,Saya temukan sebuah senjata.,effect,0,1,False


Label distribution:
label
0    250
1    250

Question types:
question
effect    254
cause     246


---
## 3. XStoryCloze — Cross-lingual Story Cloze
**Task**: Narrative commonsense (pick correct story ending)  
**Languages**: en zh es ar hi id te sw eu my ru (11)  
**Split**: eval, subsampled to 500  
**Label**: answer_right_ending ∈ {1, 2}

In [6]:
xsc_en = load("xstorycloze", "en")
print("Shape:", xsc_en.shape)
print("Columns:", list(xsc_en.columns))
show(xsc_en, 1)
print("Label distribution (answer_right_ending):")
print(xsc_en["answer_right_ending"].value_counts().to_string())

Shape: (500, 8)
Columns: ['story_id', 'input_sentence_1', 'input_sentence_2', 'input_sentence_3', 'input_sentence_4', 'sentence_quiz1', 'sentence_quiz2', 'answer_right_ending']


,story_id,input_sentence_1,input_sentence_2,input_sentence_3,input_sentence_4,sentence_quiz1,sentence_quiz2,answer_right_ending
0,fec30953-c68e-4d9d-9698-384bfe8fe857,I became a Law and Order fan in 2011.,I was recovering from a stroke.,When I got home I tried to watch every episode.,It was hard trying to binge watch 20 Year's of a show.,I think Law and Order is one of the worst shows ever made.,Eventually I watched them all.,2


Label distribution (answer_right_ending):
answer_right_ending
1    258
2    242


---
## 4. XNLI — Cross-lingual Natural Language Inference
**Task**: Textual entailment (3-class NLI)  
**Languages**: en fr es de el bg ru tr ar vi th zh hi sw ur (15)  
**Split**: test, subsampled to 500  
**Label**: 0 = entailment, 1 = neutral, 2 = contradiction

In [7]:
xnli_en = load("xnli", "en")
print("Shape:", xnli_en.shape)
print("Columns:", list(xnli_en.columns))
show(xnli_en)
print("Label distribution (0=entail, 1=neutral, 2=contradict):")
print(xnli_en["label"].value_counts().sort_index().to_string())

Shape: (500, 3)
Columns: ['premise', 'hypothesis', 'label']


,premise,hypothesis,label
0,"Well, I wasn't even thinking about that, but I was so frustrated, and, I ended up talking to him again.",I havent spoken to him again.,2
1,"Well, I wasn't even thinking about that, but I was so frustrated, and, I ended up talking to him again.",I was so upset that I just started talking to him again.,0


Label distribution (0=entail, 1=neutral, 2=contradict):
label
0    166
1    167
2    167


---
## 5. Global-MMLU-Lite — Multilingual MMLU
**Task**: 4-choice MCQA across 57 academic subjects  
**Languages**: ar bn cy de en es fr hi id it ja ko my pt sq sw yo zh (18)  
**Split**: test (~400 rows × language)  
**Label**: answer ∈ {A, B, C, D}

In [8]:
gmmlu_en = load("global_mmlu_lite", "en")
print("Shape:", gmmlu_en.shape)
print("Columns:", list(gmmlu_en.columns))
show(gmmlu_en[['question','option_a','option_b','option_c','option_d','answer','subject']], 2)
print("\nAnswer distribution:")
print(gmmlu_en["answer"].value_counts().sort_index().to_string())
print("\nSubject categories:")
print(gmmlu_en["subject_category"].value_counts().to_string())

Shape: (400, 17)
Columns: ['sample_id', 'subject', 'subject_category', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'answer', 'required_knowledge', 'time_sensitive', 'reference', 'culture', 'region', 'country', 'cultural_sensitivity_label', 'is_annotated']


,question,option_a,option_b,option_c,option_d,answer,subject
0,When traveling north from the United States into Canada you’ll see the North Star (Polaris) getting _________.,Brighter,Dimmer,Higher in the sky,Lower in the sky,C,astronomy
1,What would weigh the most on the moon?,A kilogram of feathers,Five pounds of bricks as measured on Earth,Five kilograms of feathers,A kilogram of bricks,C,astronomy



Answer distribution:
answer
A     94
B    108
C     97
D    101

Subject categories:
subject_category
Humanities         102
Social Sciences    102
Business            58
Other               56
STEM                46
Medical             36


---
## 6. TyDiQA-GoldP — Typologically Diverse QA (Gold Passage)
**Task**: Extractive QA (span prediction)  
**Languages**: english arabic bengali finnish indonesian japanese swahili korean russian telugu thai (11)  
**Split**: train (all available gold-passage examples)  
**Label**: `answers` dict with `text` (span) and byte offsets

In [9]:
tydiqa_en = load("tydiqa", "english")
print("Shape:", tydiqa_en.shape)
print("Columns:", list(tydiqa_en.columns))
print()

row = tydiqa_en.iloc[0]
print("Question :", row["question_text"])
print("Passage  :", row["passage_text"][:200], "...")
print("Answer   :", row["answers"])

print("\nExamples per language:")
lang_counts = pd.Series({
    lang: len(load("tydiqa", lang)) for lang in [
        "english","arabic","bengali","finnish","indonesian",
        "japanese","swahili","korean","russian","telugu","thai"
    ]
})
print(lang_counts.to_string())

Shape: (3696, 6)
Columns: ['id', 'language', 'document_title', 'passage_text', 'question_text', 'answers']

Question : When was quantum field theory developed?
Passage  : Quantum field theory naturally began with the study of electromagnetic interactions, as the electromagnetic field was the only known classical field as of the 1920s.[8]:1 ...
Answer   : {'text': array(['1920s'], dtype=object), 'start_byte': array([159], dtype=int32), 'limit_byte': array([164], dtype=int32)}

Examples per language:
english        3696
arabic        14805
bengali        2390
finnish        6855
indonesian     5702
japanese       4389
swahili        2755
korean         1625
russian        6490
telugu         5563
thai           3789


---
## 7. MLQA — Multilingual Question Answering
**Task**: Extractive QA (span prediction)  
**Languages**: en ar de es hi vi zh (7)  
**Split**: test  
**Label**: `answers` dict with `text` and `answer_start`

In [10]:
mlqa_en = load("mlqa", "en")
print("Shape:", mlqa_en.shape)
print("Columns:", list(mlqa_en.columns))
print()

row = mlqa_en.iloc[0]
print("Question :", row["question"])
print("Context  :", row["context"][:200], "...")
print("Answer   :", row["answers"])

print("\nRows per language:")
for lang in ["en","ar","de","es","hi","vi","zh"]:
    print(f"  {lang}: {len(load('mlqa', lang))}")

Shape: (11590, 4)
Columns: ['context', 'question', 'answers', 'id']

Question : Who analyzed the biopsies?
Context  : In 1994, five unnamed civilian contractors and the widows of contractors Walter Kasza and Robert Frost sued the USAF and the United States Environmental Protection Agency. Their suit, in which they we ...
Answer   : {'answer_start': array([457], dtype=int32), 'text': array(['Rutgers University biochemists'], dtype=object)}

Rows per language:
  en: 11590
  ar: 5335
  de: 4517
  es: 5253
  hi: 4918
  vi: 5495
  zh: 5137


---
## 8. ML-NLI-26lang — Multilingual NLI (26 Languages)
**Task**: Natural Language Inference (3-class)  
**Split**: ar_anli (500 rows sampled), all in Arabic translation  
**Columns**: premise/hypothesis in both original (en) and translated language  
**Label**: 0 = entailment, 1 = neutral, 2 = contradiction

In [11]:
mlnli = load("ml_nli_26lang", "all")
print("Shape:", mlnli.shape)
print("Columns:", list(mlnli.columns))
show(mlnli)
print("Label distribution (0=entail, 1=neutral, 2=contradict):")
print(mlnli["label"].value_counts().sort_index().to_string())

Shape: (500, 5)
Columns: ['premise_original', 'hypothesis_original', 'label', 'premise', 'hypothesis']


,premise_original,hypothesis_original,label,premise,hypothesis
0,Doom (stylized as DOOM) is a series of first-person shooter video games developed by id Software. The series focuses...,Doom is the best first person shooter game created,1,Doom (منمنمة مثل DOOM) هي سلسلة من ألعاب الفيديو مطلق النار من منظور الشخص الأول التي طورتها id Software. تركز السلس...,Doom هي أفضل لعبة مطلق النار الشخص الأول التي تم إنشاؤها
1,"WASHINGTON—U.S. military and intelligence officials are at odds over the direction of the war in Afghanistan, creati...",U.S. military and intelligence officials are in agreement about the war in afghanistan,2,قال مسؤولون أمريكيون إن المسؤولين العسكريين والاستخباراتيين الأمريكيين على خلاف حول اتجاه الحرب في أفغانستان ، مما ي...,مسؤولون عسكريون واستخباراتيون أمريكيون يتفقون على الحرب في أفغانستان


Label distribution (0=entail, 1=neutral, 2=contradict):
label
0    149
1    237
2    114


---
# Null Value Audit & Cleaning

Scan every parquet file for nulls, understand what they represent, then fix them in-place.

In [12]:
ALL_SPLITS = {
    "mgsm":             ["en","es","fr","de","ru","zh","ja","th","sw","bn","te"],
    "xcopa":            ["et","ht","id","it","qu","sw","ta","th","tr","vi","zh"],
    "xstorycloze":      ["en","zh","es","ar","hi","id","te","sw","eu","my","ru"],
    "xnli":             ["en","fr","es","de","el","bg","ru","tr","ar","vi","th","zh","hi","sw","ur"],
    "global_mmlu_lite": ["ar","bn","cy","de","en","es","fr","hi","id","it","ja","ko","my","pt","sq","sw","yo","zh"],
    "tydiqa":           ["english","arabic","bengali","finnish","indonesian","japanese","swahili","korean","russian","telugu","thai"],
    "mlqa":             ["en","ar","de","es","hi","vi","zh"],
    "ml_nli_26lang":    ["all"],
}

print("=== NULL AUDIT ACROSS ALL 85 SPLITS ===\n")
found_any = False
for ds, langs in ALL_SPLITS.items():
    for lang in langs:
        df = load(ds, lang)
        nulls = df.isnull().sum()
        nulls = nulls[nulls > 0]
        if len(nulls):
            found_any = True
            print(f"[{ds}/{lang}]  shape={df.shape}")
            for col, cnt in nulls.items():
                pct = 100 * cnt / len(df)
                print(f"  {col}: {cnt} nulls  ({pct:.1f}%)")
            print()

if not found_any:
    print("No nulls found.")

=== NULL AUDIT ACROSS ALL 85 SPLITS ===

[mgsm/en]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/es]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/fr]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/de]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/ru]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/zh]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/ja]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/th]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/sw]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 250 nulls  (100.0%)

[mgsm/bn]  shape=(250, 4)
  answer: 250 nulls  (100.0%)
  equation_solution: 

[global_mmlu_lite/my]  shape=(400, 17)


  question: 1 nulls  (0.2%)



### Findings

**MGSM — `answer` and `equation_solution` columns: 100% null across all 11 languages**  
These columns exist in the HuggingFace schema but were never populated in the downloaded split.  
`answer_number` (int32) is the ground-truth numeric answer and is fully populated.  
→ **Fix**: Drop `answer` and `equation_solution` from all MGSM parquet files.

**Global-MMLU-Lite / Myanmar (`my`) — 1 row with null `question`**  
Row `management/test/5`: the question text is missing; all 4 options and the answer letter are present, but without a question the example is unusable.  
→ **Fix**: Drop that single row from `global_mmlu_lite/my/dataset.parquet`.

In [13]:
# ── Fix 1: MGSM — drop entirely-null columns ─────────────────────────────────
DROP_COLS = ["answer", "equation_solution"]
mgsm_langs = ALL_SPLITS["mgsm"]

print("MGSM: dropping null columns and overwriting parquet files")
for lang in mgsm_langs:
    path = DATA_DIR / "mgsm" / lang / "dataset.parquet"
    df = pd.read_parquet(path)
    before = df.shape
    df = df.drop(columns=DROP_COLS)
    df.to_parquet(path, index=False)
    print(f"  mgsm/{lang}  {before} → {df.shape}")

# ── Fix 2: Global-MMLU-Lite/my — drop row with null question ─────────────────
print("\nglobal_mmlu_lite/my: dropping row with null question")
path = DATA_DIR / "global_mmlu_lite" / "my" / "dataset.parquet"
df = pd.read_parquet(path)
before = df.shape
df = df.dropna(subset=["question"]).reset_index(drop=True)
df.to_parquet(path, index=False)
print(f"  global_mmlu_lite/my  {before} → {df.shape}")

print("\nDone. Re-verifying nulls...")

MGSM: dropping null columns and overwriting parquet files
  mgsm/en  (250, 4) → (250, 2)
  mgsm/es  (250, 4) → (250, 2)
  mgsm/fr  (250, 4) → (250, 2)
  mgsm/de  (250, 4) → (250, 2)
  mgsm/ru  (250, 4) → (250, 2)
  mgsm/zh  (250, 4) → (250, 2)
  mgsm/ja  (250, 4) → (250, 2)
  mgsm/th  (250, 4) → (250, 2)
  mgsm/sw  (250, 4) → (250, 2)
  mgsm/bn  (250, 4) → (250, 2)
  mgsm/te  (250, 4) → (250, 2)

global_mmlu_lite/my: dropping row with null question
  global_mmlu_lite/my  (400, 17) → (399, 17)

Done. Re-verifying nulls...


In [14]:
# ── Post-clean null audit ──────────────────────────────────────────────────────
print("=== POST-CLEAN NULL AUDIT ===\n")
found_any = False
for ds, langs in ALL_SPLITS.items():
    for lang in langs:
        df = load(ds, lang)
        nulls = df.isnull().sum()
        nulls = nulls[nulls > 0]
        if len(nulls):
            found_any = True
            print(f"[{ds}/{lang}]")
            for col, cnt in nulls.items():
                print(f"  {col}: {cnt} nulls")

if not found_any:
    print("All 85 splits are null-free.")

=== POST-CLEAN NULL AUDIT ===



All 85 splits are null-free.


---
# Preprocessing — Unified Format

We convert every dataset into a common schema:

| field | type | description |
|---|---|---|
| `dataset` | str | dataset name |
| `language` | str | language code |
| `task_type` | str | task category |
| `id` | str | unique row id |
| `input` | str | prompt-ready input |
| `choices` | list\|None | answer options (MCQ/binary only) |
| `label` | str | ground-truth answer |

**Task types used**:
- `math_reasoning` — MGSM
- `commonsense` — XCOPA, XStoryCloze
- `nli` — XNLI, ML-NLI-26lang
- `mcqa` — Global-MMLU-Lite
- `extractive_qa` — TyDiQA-GoldP, MLQA

In [15]:
NLI_LABELS = {0: "entailment", 1: "neutral", 2: "contradiction"}

# ── MGSM ────────────────────────────────────────────────────────────────────
def preprocess_mgsm(lang: str) -> pd.DataFrame:
    df = load("mgsm", lang)
    rows = []
    for i, row in df.iterrows():
        rows.append({
            "dataset": "mgsm",
            "language": lang,
            "task_type": "math_reasoning",
            "id": f"mgsm_{lang}_{i}",
            "input": row["question"],
            "choices": None,
            "label": str(row["answer_number"]),
        })
    return pd.DataFrame(rows)


# ── XCOPA ───────────────────────────────────────────────────────────────────
def preprocess_xcopa(lang: str) -> pd.DataFrame:
    df = load("xcopa", lang)
    rows = []
    for i, row in df.iterrows():
        q_type = "Because" if row["question"] == "cause" else "Therefore"
        rows.append({
            "dataset": "xcopa",
            "language": lang,
            "task_type": "commonsense",
            "id": f"xcopa_{lang}_{row['idx']}",
            "input": f"{row['premise']} {q_type}...",
            "choices": [row["choice1"], row["choice2"]],
            "label": str(row["label"]),   # "0" or "1"
        })
    return pd.DataFrame(rows)


# ── XStoryCloze ──────────────────────────────────────────────────────────────
def preprocess_xstorycloze(lang: str) -> pd.DataFrame:
    df = load("xstorycloze", lang)
    rows = []
    for i, row in df.iterrows():
        story = " ".join([
            row["input_sentence_1"], row["input_sentence_2"],
            row["input_sentence_3"], row["input_sentence_4"],
        ])
        rows.append({
            "dataset": "xstorycloze",
            "language": lang,
            "task_type": "commonsense",
            "id": f"xstorycloze_{lang}_{row['story_id']}",
            "input": story,
            "choices": [row["sentence_quiz1"], row["sentence_quiz2"]],
            # label is 1-indexed in original; convert to 0-indexed string
            "label": str(row["answer_right_ending"] - 1),
        })
    return pd.DataFrame(rows)


# ── XNLI ─────────────────────────────────────────────────────────────────────
def preprocess_xnli(lang: str) -> pd.DataFrame:
    df = load("xnli", lang)
    rows = []
    for i, row in df.iterrows():
        rows.append({
            "dataset": "xnli",
            "language": lang,
            "task_type": "nli",
            "id": f"xnli_{lang}_{i}",
            "input": f"Premise: {row['premise']}\nHypothesis: {row['hypothesis']}",
            "choices": ["entailment", "neutral", "contradiction"],
            "label": NLI_LABELS[int(row["label"])],
        })
    return pd.DataFrame(rows)


# ── Global-MMLU-Lite ──────────────────────────────────────────────────────────
def preprocess_global_mmlu_lite(lang: str) -> pd.DataFrame:
    df = load("global_mmlu_lite", lang)
    rows = []
    for i, row in df.iterrows():
        rows.append({
            "dataset": "global_mmlu_lite",
            "language": lang,
            "task_type": "mcqa",
            "id": row["sample_id"],
            "input": (
                f"Subject: {row['subject']}\n"
                f"Question: {row['question']}"
            ),
            "choices": [row["option_a"], row["option_b"], row["option_c"], row["option_d"]],
            "label": row["answer"],   # A / B / C / D
        })
    return pd.DataFrame(rows)


# ── TyDiQA-GoldP ─────────────────────────────────────────────────────────────
def preprocess_tydiqa(lang: str) -> pd.DataFrame:
    df = load("tydiqa", lang)
    rows = []
    for i, row in df.iterrows():
        # answers is a dict with 'text' array; take first answer span
        answer_texts = row["answers"]["text"]
        answer = answer_texts[0] if len(answer_texts) > 0 else ""
        rows.append({
            "dataset": "tydiqa",
            "language": lang,
            "task_type": "extractive_qa",
            "id": row["id"],
            "input": f"Passage: {row['passage_text']}\nQuestion: {row['question_text']}",
            "choices": None,
            "label": answer,
        })
    return pd.DataFrame(rows)


# ── MLQA ──────────────────────────────────────────────────────────────────────
def preprocess_mlqa(lang: str) -> pd.DataFrame:
    df = load("mlqa", lang)
    rows = []
    for i, row in df.iterrows():
        answer_texts = row["answers"]["text"]
        answer = answer_texts[0] if len(answer_texts) > 0 else ""
        rows.append({
            "dataset": "mlqa",
            "language": lang,
            "task_type": "extractive_qa",
            "id": row["id"],
            "input": f"Context: {row['context']}\nQuestion: {row['question']}",
            "choices": None,
            "label": answer,
        })
    return pd.DataFrame(rows)


# ── ML-NLI-26lang ─────────────────────────────────────────────────────────────
def preprocess_ml_nli_26lang() -> pd.DataFrame:
    df = load("ml_nli_26lang", "all")
    rows = []
    for i, row in df.iterrows():
        rows.append({
            "dataset": "ml_nli_26lang",
            "language": "ar",    # sampled split is Arabic
            "task_type": "nli",
            "id": f"ml_nli_26lang_ar_{i}",
            "input": f"Premise: {row['premise']}\nHypothesis: {row['hypothesis']}",
            "choices": ["entailment", "neutral", "contradiction"],
            "label": NLI_LABELS[int(row["label"])],
        })
    return pd.DataFrame(rows)


print("Preprocessors defined.")

Preprocessors defined.


In [16]:
# Spot-check one language per dataset
print("=== MGSM (en) ===")
show(preprocess_mgsm("en"), 2)

print("\n=== XCOPA (id) ===")
show(preprocess_xcopa("id"), 2)

print("\n=== XStoryCloze (en) ===")
show(preprocess_xstorycloze("en"), 2)

print("\n=== XNLI (en) ===")
show(preprocess_xnli("en"), 2)

print("\n=== Global-MMLU-Lite (en) ===")
show(preprocess_global_mmlu_lite("en"), 2)

print("\n=== TyDiQA (english) ===")
show(preprocess_tydiqa("english"), 2)

print("\n=== MLQA (en) ===")
show(preprocess_mlqa("en"), 2)

print("\n=== ML-NLI-26lang (all/ar) ===")
show(preprocess_ml_nli_26lang(), 2)

=== MGSM (en) ===


,dataset,language,task_type,id,input,choices,label
0,mgsm,en,math_reasoning,mgsm_en_0,Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends ever...,None,18
1,mgsm,en,math_reasoning,mgsm_en_1,A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?,None,3



=== XCOPA (id) ===


,dataset,language,task_type,id,input,choices,label
0,xcopa,id,commonsense,xcopa_id_0,Barang itu dikemas dalam bungkus gelembung. Because...,"[Barang itu rapuh., Barang itu kecil.]",0
1,xcopa,id,commonsense,xcopa_id_1,Saya kosongkan kantong saya. Therefore...,"[Saya ambil sebuah potongan tiket., Saya temukan sebuah senjata.]",0



=== XStoryCloze (en) ===


...

,dataset,language,task_type,id,input,choices,label
0,tydiqa,english,extractive_qa,-9041575374418655524-12,"Passage: Quantum field theory naturally began with the study of electromagnetic interactions, as the electromagnetic...",None,1920s
1,tydiqa,english,extractive_qa,1971662537536642354-0,Passage: The Nobel Prize in Literature (Swedish: Nobelpriset i litteratur) is awarded annually by the Swedish Academ...,None,Sully Prudhomme



=== MLQA (en) ===


,dataset,language,task_type,id,input,choices,label
0,mlqa,en,extractive_qa,a4968ca8a18de16aa3859be760e43dbd3af3fce9,"Context: In 1994, five unnamed civilian contractors and the widows of contractors Walter Kasza and Robert Frost sued...",None,Rutgers University biochemists
1,mlqa,en,extractive_qa,f251ea56c4f1aa1df270137f7e6d89c0cc1b6ef4,"Context: In 1994, five unnamed civilian contractors and the widows of contractors Walter Kasza and Robert Frost sued...",None,George Washington University law professor Jonathan Turley



=== ML-NLI-26lang (all/ar) ===


,dataset,language,task_type,id,input,choices,label
0,ml_nli_26lang,ar,nli,ml_nli_26lang_ar_0,Premise: Doom (منمنمة مثل DOOM) هي سلسلة من ألعاب الفيديو مطلق النار من منظور الشخص الأول التي طورتها id Software. ت...,"[entailment, neutral, contradiction]",neutral
1,ml_nli_26lang,ar,nli,ml_nli_26lang_ar_1,Premise: قال مسؤولون أمريكيون إن المسؤولين العسكريين والاستخباراتيين الأمريكيين على خلاف حول اتجاه الحرب في أفغانستا...,"[entailment, neutral, contradiction]",contradiction


## Build the full preprocessed corpus

Iterate over all languages for each dataset and concatenate into one DataFrame.

In [17]:
MGSM_LANGS = ["en","es","fr","de","ru","zh","ja","th","sw","bn","te"]
XCOPA_LANGS = ["et","ht","id","it","qu","sw","ta","th","tr","vi","zh"]
XSTORYCLOZE_LANGS = ["en","zh","es","ar","hi","id","te","sw","eu","my","ru"]
XNLI_LANGS = ["en","fr","es","de","el","bg","ru","tr","ar","vi","th","zh","hi","sw","ur"]
GMMLU_LANGS = ["ar","bn","cy","de","en","es","fr","hi","id","it","ja","ko","my","pt","sq","sw","yo","zh"]
TYDIQA_LANGS = ["english","arabic","bengali","finnish","indonesian","japanese","swahili","korean","russian","telugu","thai"]
MLQA_LANGS = ["en","ar","de","es","hi","vi","zh"]

parts = []

for lang in MGSM_LANGS:
    parts.append(preprocess_mgsm(lang))
for lang in XCOPA_LANGS:
    parts.append(preprocess_xcopa(lang))
for lang in XSTORYCLOZE_LANGS:
    parts.append(preprocess_xstorycloze(lang))
for lang in XNLI_LANGS:
    parts.append(preprocess_xnli(lang))
for lang in GMMLU_LANGS:
    parts.append(preprocess_global_mmlu_lite(lang))
for lang in TYDIQA_LANGS:
    parts.append(preprocess_tydiqa(lang))
for lang in MLQA_LANGS:
    parts.append(preprocess_mlqa(lang))
parts.append(preprocess_ml_nli_26lang())

corpus = pd.concat(parts, ignore_index=True)
print("Total examples:", len(corpus))
display(corpus.groupby(["dataset","task_type"])["id"].count().rename("examples"))

Total examples: 129253


dataset           task_type     
global_mmlu_lite  mcqa               7199
mgsm              math_reasoning     2750
ml_nli_26lang     nli                 500
mlqa              extractive_qa     42245
tydiqa            extractive_qa     58059
xcopa             commonsense        5500
xnli              nli                7500
xstorycloze       commonsense        5500
Name: examples, dtype: int64

In [18]:
print("Corpus shape :", corpus.shape)
print("Columns      :", list(corpus.columns))
print("Null values  :")
print(corpus.isnull().sum())
print("\nSample rows:")
show(corpus.sample(5, random_state=42))

Corpus shape : (129253, 7)
Columns      : ['dataset', 'language', 'task_type', 'id', 'input', 'choices', 'label']
Null values  :
dataset           0
language          0
task_type         0
id                0
input             0
choices      103054
label             0
dtype: int64

Sample rows:


,dataset,language,task_type,id,input,choices,label
39251,tydiqa,arabic,extractive_qa,-1747958826820552357-5,Passage: تشكل الحرس الثوري في 5 أيار/مايو 1979[15][16] في أعقاب الثورة الإسلامية وذلك في محاولة لتوحيد العديد من الق...,None,5 أيار/مايو 1979
69851,tydiqa,korean,extractive_qa,905946218190344235-4,"Passage: \n1685년 거리의 악사(바이올린 주자) 요한 암브로지우스(1645~1695)의 막내아들로서 튀링겐 지방의 아이제나흐에서 태어나, 3월 23일에 그 곳의 성(聖) 게오르크 교회에서 세례를 받...",None,이제나흐


## Save the unified corpus

In [19]:
# Convert list column to JSON strings for Parquet compatibility
import json as _json

corpus_to_save = corpus.copy()
corpus_to_save["choices"] = corpus_to_save["choices"].apply(
    lambda x: _json.dumps(x, ensure_ascii=False) if x is not None else None
)

out_path = DATA_DIR / "corpus.parquet"
corpus_to_save.to_parquet(out_path, index=False)
size_mb = out_path.stat().st_size / 1_048_576
print(f"Saved {len(corpus_to_save):,} rows → {out_path}  ({size_mb:.2f} MB)")

# Reload to verify round-trip
check = pd.read_parquet(out_path)
assert len(check) == len(corpus_to_save)
print("Round-trip check passed.")

Saved 129,253 rows → data/corpus.parquet  (59.66 MB)
Round-trip check passed.
